# LabView process data (building 67, box 5)

`LVData` holds the process log of the high-pressure setup: one variable per
channel along `tos`, each with its unit, group, species and location in the
attributes. There is no LabView fixture under `tests/`, so this notebook reads
the excerpt in `data/drifts_run/labview/` (several export files of one run).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import lab_data_parser as ldp

In [ ]:
LABVIEW = Path("data") / "drifts_run" / "labview"
sorted(p.name for p in LABVIEW.glob("*.txt"))

## Read a directory

All files in the folder are concatenated in time. Without a `tos_start` the
origin is the first row; here it is set to the start of the reaction step.

In [ ]:
tos_start = pd.Timestamp("2026-03-26 15:27:00", tz="Europe/Amsterdam")
lv_raw = ldp.LVData.from_b67_box5_txt(LABVIEW, tos_start=tos_start)
lv_raw

In [ ]:
for channel in lv_raw.channels:
    attrs = lv_raw.ds[channel].attrs
    print(f"{channel:15s} {attrs.get('unit') or '':8s} {attrs.get('group') or '':12s} {attrs.get('species') or ''}")

## Resample and smooth

The log is written every 5 s. `resample` bins it onto a coarser grid,
`smooth_moving` applies a centred moving average.

In [ ]:
lv = lv_raw.resample(60).smooth_moving(5)
print(lv_raw.n_samples, "->", lv.n_samples, "samples")
print(lv.format_history())

In [ ]:
fig, (ax_t, ax_f) = plt.subplots(2, 1, figsize=(7, 5), sharex=True)
ax_t.plot(lv.tos / 3600, lv.get_channel("Reactor T PV"))
ax_t.set_ylabel(f"temperature / {lv.get_channel_unit('Reactor T PV')}")
for channel in ("F2 He PV", "F2 H2 PV", "F2 CO2 PV"):
    ax_f.plot(lv.tos / 3600, lv.get_channel(channel), label=channel.lower())
ax_f.set_ylabel(f"flow / {lv.get_channel_unit('F2 H2 PV')}")
ax_f.set_xlabel("time on stream / h")
ax_f.legend()
plt.show()

## Species names

`rename_species` maps the species attribute of every channel onto the standard
naming convention. The parser never renames on read.

In [ ]:
renamed = lv_raw.rename_species()
{channel: renamed.ds[channel].attrs.get("species") for channel in renamed.channels if renamed.ds[channel].attrs.get("species")}

## Groups and export

In [ ]:
flows = lv_raw.select_group("flow")
print(flows.channels)
flows.to_dataframe().head()